Imports:

In [38]:
import jax
import jax.numpy as jnp
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import seaborn as sns


Load Dataset:

In [39]:
data = load_breast_cancer()
X = data.data
y = data.target
print(X.shape, Y.shape)

(569, 30) (569,)


Standarize Data:

In [40]:
scaler = StandardScaler()
X = scaler.fit_transform(X)
X

array([[ 1.09706398, -2.07333501,  1.26993369, ...,  2.29607613,
         2.75062224,  1.93701461],
       [ 1.82982061, -0.35363241,  1.68595471, ...,  1.0870843 ,
        -0.24388967,  0.28118999],
       [ 1.57988811,  0.45618695,  1.56650313, ...,  1.95500035,
         1.152255  ,  0.20139121],
       ...,
       [ 0.70228425,  2.0455738 ,  0.67267578, ...,  0.41406869,
        -1.10454895, -0.31840916],
       [ 1.83834103,  2.33645719,  1.98252415, ...,  2.28998549,
         1.91908301,  2.21963528],
       [-1.80840125,  1.22179204, -1.81438851, ..., -1.74506282,
        -0.04813821, -0.75120669]], shape=(569, 30))

Split Dataset:

In [41]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, shuffle = True)
"""random_state sets the seed for the random shuffle used to split your data. With random_state=42, you’ll get
 the same train/test split each time you run the code, which makes results reproducible. The value 42 is 
 arbitrary—you can use another integer. If you omit random_state, the split will usually be
 different each run."""
 
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(455, 30) (114, 30) (455,) (114,)


Define Model:

In [42]:
def sigmoid(x):
    return jax.nn.sigmoid(x)

def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1 - s)

Define Architecture, weights, biases and parameters:

In [43]:
# Define the neural network architecture
input_neurons = X_train.shape[1]
hidden_neurons = 10
output_neurons = 1

# Initialize weights and biases
key = jax.random.PRNGKey(42)
# Creates a JAX random-number key using the seed 42. This makes random values reproducible:
#  rerunning the code with the same key gives the same values.

weights_input_hidden = jax.random.uniform(key, shape=(input_neurons, hidden_neurons))
# Creates a matrix of random weights connecting the input layer to the hidden layer.
#  Its shape means there’s one weight for every input-to-hidden-neuron connection.
#  uniform draws values between 0 and 1 by default.

biases_hidden = jnp.zeros((1, hidden_neurons))
weights_hidden_output = jax.random.uniform(key, shape=(hidden_neurons, output_neurons))
biases_output = jnp.zeros((1, output_neurons))

# Hyperparameters
learning_rate = 0.001
epochs = 10000

Train Data into Neural Network:

In [44]:
# Training the neural network
for epoch in range(epochs):
    # forward propagation
    hidden_inputs = jnp.dot(X_train, weights_input_hidden) + biases_hidden
    hidden_outputs = sigmoid(hidden_inputs)

    final_inputs = jnp.dot(hidden_outputs, weights_hidden_output) + biases_output
    final_outputs = sigmoid(final_inputs)

    # Calculate the error
    error = y_train.reshape(-1, 1) - final_outputs
    """Reshapes the labels into a column so their shape matches the predictions,
     then subtracts each prediction from its true label. Positive error means
      the prediction was too low; negative error means it was too high."""

    # Backpropagation
    output_error_term = error * sigmoid_derivative(final_outputs)
    """Combines the output error with the sigmoid derivative. 
    This estimates how much each output neuron’s input contributed to its prediction error."""

    hidden_error = output_error_term.dot(weights_hidden_output.T)
    """Passes the output-layer error backward through the hidden-to-output weights to estimate
     the error associated with each hidden neuron."""
     
    hidden_error_term = hidden_error * sigmoid_derivative(hidden_outputs)
    # Applies the sigmoid derivative for the hidden layer, giving the adjustment signal for each hidden neuron.

    # Update weights and biases
    weights_hidden_output += hidden_outputs.T.dot(output_error_term) * learning_rate
    biases_output += jnp.sum(output_error_term, axis= 0, keepdims= True) * learning_rate
    weights_input_hidden += X_train.T.dot(hidden_error_term) * learning_rate
    biases_hidden += jnp.sum(hidden_error_term, axis= 0 , keepdims= True) * learning_rate


Test Nueral Network:

In [49]:
# Testing the neural network
hidden_layer_input = jnp.dot(X_test, weights_input_hidden) + biases_hidden
hidden_layer_output = sigmoid(hidden_layer_input)
output = jnp.dot(hidden_layer_output, weights_hidden_output) + biases_output
predicted_output = sigmoid(output)

# convert predicted output to binary labels (0 or 1)
predicted_labels = jnp.round(predicted_output).flatten()
# flatten converts the result to a one dimensional array

# calculate the accuracy
accuracy = jnp.mean(predicted_labels == y_test)
print("Accuracy: ", accuracy)

Accuracy:  0.9736842


Compare with sklearn:

In [46]:
from sklearn.neural_network import MLPClassifier

clf = MLPClassifier(random_state=42, solver="lbfgs", max_iter=500).fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(y_pred)

[1 0 0 1 1 0 0 0 1 1 1 0 1 0 1 0 1 1 1 0 1 1 0 1 1 1 1 1 1 0 1 1 1 1 1 1 0
 1 0 1 1 0 1 1 1 0 1 1 1 1 0 0 1 1 1 1 1 0 0 1 1 0 0 1 1 1 0 0 1 1 0 0 1 0
 1 1 1 1 1 1 0 1 0 0 0 0 0 0 1 1 1 0 1 1 1 1 0 0 1 0 0 1 0 0 1 1 1 0 1 1 0
 1 1 0]


In [47]:
# Calculate accuracy
correct = []

for i in range(len(y_test)):
    if y_pred[i] == y_test[i]:
        right = 1
    else: 
        right = 0
    correct.append(right)
print(correct)

acc = sum(correct)/len(y_test)
print(acc)

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
0.9649122807017544
